# Оператор зала: старт без GPU

Задача, определения счётчиков и правила — в `TASK.md`. Этот ноутбук читает только пакет участника. Внешние данные и веса не скачивает.


In [ ]:
from pathlib import Path
import os, sys

if os.environ.get("TABLE_STATE_DATA"):
    DATA = Path(os.environ["TABLE_STATE_DATA"])
elif Path("train_labels.csv").is_file():
    DATA = Path.cwd()
else:
    candidates = [p.parent for p in Path("/kaggle/input").rglob("train_labels.csv")
                  if (p.parent / "tables.csv").is_file() and (p.parent / "data_card.json").is_file()]
    if len(candidates) != 1:
        raise ValueError("Распакуйте данные и укажите папку через TABLE_STATE_DATA.")
    DATA = candidates[0]
OUT = Path(os.environ.get("TABLE_STATE_OUT", "/kaggle/working" if Path("/kaggle").exists() else "."))
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(DATA.resolve()))
from check_data import check
from baseline import make_features, run
print(check(DATA))


## Локальная проверка по дням

Модель видит метки 25 октября и проверяется на 26 октября. Не делите столы одного кадра случайно между train и validation.


In [ ]:
features = make_features(DATA)
validation = {}
for mode in ["median", "detector", "calibrated"]:
    print("\nMODE:", mode)
    _, result = run(DATA, mode=mode, validate=True, features=features)
    validation[mode] = result["score"]
print("Validation scores:", validation)


## Сабмит

Начните с простого геометрического бейзлайна. Для другого варианта поменяйте `MODE`, проверив его локально. Без `--validate` модель использует все train-метки.


In [ ]:
MODE = "detector"
SUBMISSION = OUT / "submission.csv"
prediction, config = run(DATA, out=SUBMISSION, mode=MODE, features=features)
print(prediction.head())


## Пример применения

Очередь подсказок оператору строится из прогнозов. Подсказка «проверить посуду» не означает, что событие уборки было распознано или размечено.


In [ ]:
from preview import preview
from PIL import Image
preview(DATA, SUBMISSION, OUT / "preview.jpg", count=4)
if "display" in globals():
    display(Image.open(OUT / "preview.jpg"))
else:
    print("Preview saved to", OUT / "preview.jpg")
